# StormEngine V8 — Stage-1 refinement

Continue the published sigma=0.10 baseline to convergence and run a controlled sigma screen. This notebook does not read 2017 or the August 2026 operational week. Keep all run switches false until paths and checkpoint hashes are verified.

In [ ]:
from pathlib import Path
import hashlib, subprocess, sys

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
BASELINE = REPO / 'artifacts' / 'v8_spatial_pretraining'
BEST = BASELINE / 'best.pt'
LAST = BASELINE / 'last.pt'
EXPECTED_BEST_SHA256 = '23cda69cb9481b21a8dbfc236ffc6915e81e06367081db982d91d69e86ab407e'
assert BEST.is_file() and LAST.is_file(), 'Copy/retain both Stage-1 checkpoints first'
actual = hashlib.sha256(BEST.read_bytes()).hexdigest()
assert actual == EXPECTED_BEST_SHA256, (actual, EXPECTED_BEST_SHA256)
DEVICE = 'cuda'
print('Repository:', REPO)
print('Baseline best checkpoint verified:', actual)

In [ ]:
def run_live(arguments):
    command = [str(item) for item in arguments]
    print('Running:', ' '.join(command), flush=True)
    completed = subprocess.run(command, cwd=REPO, check=True)
    return completed.returncode

subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'),
                'preflight', '--device', DEVICE,
                '--config', str(REPO / 'configs' / 'v8_reconstruction_continue.yaml')],
               cwd=REPO, check=True)

## 1. Continue sigma=0.10 to convergence

This targets epoch 90 total, so an epoch-60 checkpoint runs at most 30 more epochs. It writes to a new directory and preserves the published baseline.

In [ ]:
RUN_CONTINUATION = False
if RUN_CONTINUATION:
    run_live([sys.executable, '-u', REPO / 'scripts' / 'train_v8_reconstruction.py',
              'train', '--device', DEVICE,
              '--config', REPO / 'configs' / 'v8_reconstruction_continue.yaml',
              '--resume', LAST])

## 2. Controlled sigma screen

All three candidates start from scratch with the same seed and capped budget. These are screening results, not reportable final models.

In [ ]:
RUN_SIGMA_SCREENS = False
sigma_configs = [
    REPO / 'configs' / 'v8_reconstruction_sigma007.yaml',
    REPO / 'configs' / 'v8_reconstruction_sigma010.yaml',
    REPO / 'configs' / 'v8_reconstruction_sigma015.yaml',
]
if RUN_SIGMA_SCREENS:
    for config in sigma_configs:
        run_live([sys.executable, '-u', REPO / 'scripts' / 'train_v8_reconstruction.py',
                  'screen', '--device', DEVICE, '--config', config])

In [ ]:
screen_dirs = [
    REPO / 'artifacts' / 'v8_spatial_screen_sigma007',
    REPO / 'artifacts' / 'v8_spatial_screen_sigma010',
    REPO / 'artifacts' / 'v8_spatial_screen_sigma015',
]
if all((path / 'screen_summary.json').is_file() for path in screen_dirs):
    run_live([sys.executable, '-u', REPO / 'scripts' / 'compare_v8_spatial_screens.py',
              *screen_dirs, '--output', REPO / 'artifacts' / 'v8_spatial_sigma_comparison.json'])
else:
    print('Run all three screens before comparison.')

## Stop here

Send the continuation `train_summary.json`, all three `screen_summary.json` files, and `v8_spatial_sigma_comparison.json` for review. Do not start a full non-baseline candidate or Processor training until the screening evidence is reviewed.